# Inteligência Artificial (PCO118A) — Aula 7 — Laboratório: Poda Alfa-Beta
**Prof. Carlos Costa Cedro** · Bacharelado em Ciência da Computação · PUCPR

**GABARITO DO PROFESSOR.** Todas as lacunas resolvidas e conferidas por execução.

Este laboratório parte do **mesmo código da Aula 6**. A modificação é de dois parâmetros e duas linhas, e o objetivo é que vocês vejam isso nas próprias mãos: o valor devolvido não muda, e o custo despenca.

In [1]:
# ============================================================
# CÓDIGO DA AULA 6 — fornecido, para o notebook ser autocontido
# ============================================================
import time, random
from functools import lru_cache

LINHAS = [(0,1,2),(3,4,5),(6,7,8),(0,3,6),(1,4,7),(2,5,8),(0,4,8),(2,4,6)]
VAZIO  = ' ' * 9
NOMES  = {0:'canto', 1:'lateral', 2:'canto', 3:'lateral', 4:'centro',
          5:'lateral', 6:'canto', 7:'lateral', 8:'canto'}

def venceu(b, p):    return any(all(b[i] == p for i in l) for l in LINHAS)
def eh_terminal(b):  return venceu(b,'X') or venceu(b,'O') or ' ' not in b
def utilidade(b):    return 1 if venceu(b,'X') else (-1 if venceu(b,'O') else 0)
def jogadas(b):      return [i for i in range(9) if b[i] == ' ']
def resultado(b,i,p):return b[:i] + p + b[i+1:]
def oponente(p):     return 'O' if p == 'X' else 'X'

# Números medidos por você na Aula 6, para conferência
MINIMAX_TOTAL = 549946
MINIMAX_POR_JOGADA = {0:59705, 1:63905, 2:59705, 3:63905, 4:55505,
                      5:63905, 6:59705, 7:63905, 8:59705}
print('Setup da Aula 6 carregado.')

Setup da Aula 6 carregado.


## Lacuna 1 — acrescentar α e β

Parta das suas funções da Aula 6 e acrescente **dois parâmetros e duas linhas**. Nada mais.

- Em `valor_max`: depois de atualizar `v`, se `v >= beta` interrompa e devolva; senão
  `alfa = max(alfa, v)`.
- Em `valor_min`: depois de atualizar `v`, se `v <= alfa` interrompa e devolva; senão
  `beta = min(beta, v)`.

> **Atenção.** `alfa` e `beta` são parâmetros locais. Se você guardá-los em variável
> global ou em atributo de objeto, eles vazam entre chamadas irmãs, o algoritmo poda
> demais e às vezes ainda devolve o valor certo, o que torna o defeito difícil de achar.
>
> **Atenção 2.** O teste de corte vem **antes** da atualização de `alfa` ou `beta`.

In [2]:
NOS = 0

def valor_max(b, alfa, beta, ordem=range(9)):
    global NOS
    NOS += 1
    if eh_terminal(b):
        return utilidade(b), None
    v, melhor = -2, None
    for i in [k for k in ordem if b[k] == ' ']:
        v2, _ = valor_min(resultado(b, i, 'X'), alfa, beta, ordem)
        if v2 > v:
            v, melhor = v2, i
        if v >= beta:
            return v, melhor          # corte beta
        alfa = max(alfa, v)
    return v, melhor

def valor_min(b, alfa, beta, ordem=range(9)):
    global NOS
    NOS += 1
    if eh_terminal(b):
        return utilidade(b), None
    v, melhor = 2, None
    for i in [k for k in ordem if b[k] == ' ']:
        v2, _ = valor_max(resultado(b, i, 'O'), alfa, beta, ordem)
        if v2 < v:
            v, melhor = v2, i
        if v <= alfa:
            return v, melhor          # corte alfa
        beta = min(beta, v)
    return v, melhor

def alfa_beta(b, p, ordem=range(9)):
    f = valor_max if p == 'X' else valor_min
    return f(b, -2, 2, ordem)

## Lacuna 2 — o teste que importa

Meça o valor e a contagem de nós a partir do tabuleiro vazio, e compare com a Aula 6.

**O valor tem de continuar sendo 0.** Se mudou, a implementação está errada: pelo
Teorema 1 da Aula 7, alfa-beta devolve exatamente o mesmo valor que Minimax.

In [3]:
NOS = 0
t = time.time()
valor, jogada = alfa_beta(VAZIO, 'X')
print(f'valor            = {valor}   (Aula 6: 0)')
print(f'nós visitados    = {NOS}   (Aula 6: {MINIMAX_TOTAL})')
print(f'redução          = {MINIMAX_TOTAL / NOS:.1f}x')
print(f'tempo            = {time.time() - t:.2f} s')

assert valor == 0, 'o Teorema 1 foi violado: há defeito na implementação'
assert NOS == 18297, f'esperado 18297 nós com a ordem natural, obtido {NOS}'
print('\nLacuna 2: OK')

valor            = 0   (Aula 6: 0)
nós visitados    = 18297   (Aula 6: 549946)
redução          = 30.1x
tempo            = 0.13 s

Lacuna 2: OK


## Lacuna 3 — completar a tabela da Aula 6

Repita a medição por primeira jogada e complete a tabela que você trouxe.

In [4]:
ab_por_jogada = {}
for i in range(9):
    b = resultado(VAZIO, i, 'X')
    NOS = 0
    alfa_beta(b, 'O')
    ab_por_jogada[i] = NOS

In [5]:
print(f"{'casa':>5} {'posição':>9} {'minimax':>9} {'alfa-beta':>10} {'razão':>7}")
for i in range(9):
    m, a = MINIMAX_POR_JOGADA[i], ab_por_jogada[i]
    print(f'{i:>5} {NOMES[i]:>9} {m:>9} {a:>10} {m/a:>6.1f}x')

assert (ab_por_jogada[4], ab_por_jogada[0], ab_por_jogada[1]) == (2316, 2338, 2869)
print('\nLacuna 3: OK')

 casa   posição   minimax  alfa-beta   razão
    0     canto     59705       2338   25.5x
    1   lateral     63905       2869   22.3x
    2     canto     59705       3275   18.2x
    3   lateral     63905       3574   17.9x
    4    centro     55505       2316   24.0x
    5   lateral     63905       3590   17.8x
    6     canto     59705       3809   15.7x
    7   lateral     63905       4981   12.8x
    8     canto     59705       3957   15.1x

Lacuna 3: OK


### Para o relatório

As reduções por jogada ficam entre 22 e 26 vezes, mas a redução global da lacuna 2 é de
**30 vezes**. **Por que a redução global é maior que qualquer redução individual?**

> Pista: na busca a partir do tabuleiro vazio, o que a segunda jogada da raiz já sabe
> que a busca isolada daquela subárvore não sabe?

## Lacuna 4 — a ordem muda o custo, nunca a resposta

Meça pelo menos três ordens de geração de jogadas. Registre valor e nós de cada uma.

In [6]:
ORDENS = {
    'natural'          : list(range(9)),
    'invertida'        : list(range(8, -1, -1)),
    'centro_cantos_lados': [4, 0, 2, 6, 8, 1, 3, 5, 7],
}

resultados = {}
for nome, ordem in ORDENS.items():
    NOS = 0
    v, _ = alfa_beta(VAZIO, 'X', ordem)
    resultados[nome] = (v, NOS)
    print(f'{nome:<22} valor = {v}   nós = {NOS:>6}   redução = {MINIMAX_TOTAL/NOS:>5.1f}x')

valores = {v for v, _ in resultados.values()}
print(f'\nvalores distintos devolvidos: {valores}')
assert valores == {0}, 'o Teorema 1 foi violado'
assert resultados['centro_cantos_lados'][1] == 7275
print('Lacuna 4: OK — um único valor, três custos')

natural                valor = 0   nós =  18297   redução =  30.1x


invertida              valor = 0   nós =  18297   redução =  30.1x
centro_cantos_lados    valor = 0   nós =   7275   redução =  75.6x

valores distintos devolvidos: {0}
Lacuna 4: OK — um único valor, três custos


### Para o relatório

1. Quantos valores distintos apareceram na coluna do valor? Relacione com o Teorema 1.
2. Registre a menor e a maior contagem e a razão entre elas. Compare com a razão de
   7 para 12 obtida na atividade 2 da aula. A ordem natural está mais perto do melhor
   ou do pior caso?
3. A ordem `centro_cantos_lados` aproxima a ordenação perfeita usando conhecimento sobre
   o jogo, e não sobre a árvore. **Por que a ordenação perfeita de verdade é impossível
   de obter antes da busca?**

## Lacuna 5 — cortar antes das folhas

Aqui a segunda hipótese cai. Substitua `eh_terminal` por um teste de corte por
profundidade e `utilidade` por uma função de avaliação. **A partir deste ponto, os
Teoremas 1 e 2 da Aula 6 deixam de valer.**

São fornecidas duas funções de avaliação, uma razoável e uma deliberadamente ruim.
Implemente `alfa_beta_corte` e compare os dois agentes contra o agente de busca completa.

Este é o piloto do TDE2, e é aberto: os números não estão travados por `assert`.

In [7]:
# ============================================================
# FORNECIDO — duas funções de avaliação e o agente de busca completa
# ============================================================
def eval_boa(b):
    """Linhas ainda abertas para X menos linhas ainda abertas para O."""
    if eh_terminal(b):
        return utilidade(b) * 10
    abertas_x = sum(1 for l in LINHAS if all(b[i] in ' X' for i in l))
    abertas_o = sum(1 for l in LINHAS if all(b[i] in ' O' for i in l))
    return abertas_x - abertas_o

def eval_ruim(b):
    """Só conta marcas no tabuleiro. Não diz nada sobre ameaças."""
    if eh_terminal(b):
        return utilidade(b) * 10
    return b.count('X') - b.count('O')

@lru_cache(maxsize=None)
def valor_completo(b, p):
    if eh_terminal(b): return utilidade(b)
    vs = [valor_completo(resultado(b, i, p), oponente(p)) for i in jogadas(b)]
    return max(vs) if p == 'X' else min(vs)

def jogada_completa(b, p):
    f = max if p == 'X' else min
    return f(jogadas(b), key=lambda i: valor_completo(resultado(b, i, p), oponente(p)))

In [8]:
def alfa_beta_corte(b, p, alfa, beta, d, ev):
    if eh_terminal(b) or d == 0:
        return ev(b)
    if p == 'X':
        v = -99
        for i in jogadas(b):
            v = max(v, alfa_beta_corte(resultado(b, i, 'X'), 'O', alfa, beta, d-1, ev))
            if v >= beta: return v
            alfa = max(alfa, v)
        return v
    v = 99
    for i in jogadas(b):
        v = min(v, alfa_beta_corte(resultado(b, i, 'O'), 'X', alfa, beta, d-1, ev))
        if v <= alfa: return v
        beta = min(beta, v)
    return v

def jogada_com_corte(b, p, d, ev):
    f = max if p == 'X' else min
    return f(jogadas(b),
             key=lambda i: alfa_beta_corte(resultado(b, i, p), oponente(p), -99, 99, d-1, ev))

In [9]:
# Dois agentes determinísticos jogam sempre a MESMA partida. Para obter variedade,
# a primeira jogada é sorteada. Este detalhe metodológico vale para o TDE2.
def duelo(d, ev, n=36, semente=3):
    rng = random.Random(semente)
    r = {'corte venceu': 0, 'empate': 0, 'completo venceu': 0}
    for _ in range(n):
        b, p, primeira = VAZIO, 'X', True
        while not eh_terminal(b):
            if primeira:
                i = rng.choice(jogadas(b)); primeira = False
            elif p == 'X':
                i = jogada_com_corte(b, p, d, ev)
            else:
                i = jogada_completa(b, p)
            b = resultado(b, i, p); p = oponente(p)
        u = utilidade(b)
        r['corte venceu' if u > 0 else ('completo venceu' if u < 0 else 'empate')] += 1
    return r

print('X = agente com corte, O = agente de busca completa, 36 partidas\n')
for nome, ev in [('eval_boa', eval_boa), ('eval_ruim', eval_ruim)]:
    for d in (2, 4, 6):
        print(f'  {nome:<10} d={d}: {duelo(d, ev)}')
    print()

X = agente com corte, O = agente de busca completa, 36 partidas



  eval_boa   d=2: {'corte venceu': 0, 'empate': 36, 'completo venceu': 0}


  eval_boa   d=4: {'corte venceu': 0, 'empate': 36, 'completo venceu': 0}


  eval_boa   d=6: {'corte venceu': 0, 'empate': 36, 'completo venceu': 0}

  eval_ruim  d=2: {'corte venceu': 0, 'empate': 31, 'completo venceu': 5}


  eval_ruim  d=4: {'corte venceu': 0, 'empate': 34, 'completo venceu': 2}


  eval_ruim  d=6: {'corte venceu': 0, 'empate': 36, 'completo venceu': 0}



### Para o relatório

1. Com `eval_boa`, o agente com corte **não perde em nenhuma profundidade**. Com
   `eval_ruim`, ele perde em `d=2` e `d=4`, e para de perder em `d=6`. Explique o papel
   da profundidade em compensar uma função de avaliação ruim.
2. Se o agente com corte perde para o agente completo, **isso contradiz o Teorema 1 da
   Aula 7?** Justifique nomeando qual hipótese cada agente abandonou. Cuidado: são
   teoremas diferentes, e confundi-los é o erro que esta pergunta detecta.
3. Com `eval_boa` não foi possível observar o efeito horizonte neste jogo. **Por quê?**
   O que isso sugere sobre a escolha do jogo no TDE2?

---
## Entrega

- Respostas às perguntas das lacunas 3, 4 e 5.
- A tabela completa da lacuna 3, com as colunas de minimax e de alfa-beta lado a lado.
- Toda medição de nós acompanhada do tabuleiro inicial e da ordem de jogadas usada.
- Uma frase declarando, para o agente da lacuna 5, **quais garantias ele não tem**.

Este notebook é o piloto do **TDE2 (Busca Competitiva)**, com execução entre 30/09 e
07/10 e apresentação em 07/10. As três variáveis a controlar já apareceram aqui:
profundidade de corte, qualidade da ordenação e qualidade da função de avaliação.